# Global Salary Data by Skill & Role (620k jobs): starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/skill_salary_percentiles_global_sample.csv', recursive=True) or ['skill_salary_percentiles_global_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
df.groupby('country_name').salary_mid_usd.agg(['count', 'median']).query('count >= 30').sort_values('median', ascending=False)

In [ ]:
sk = df.dropna(subset=['skills']).assign(skill=lambda d: d.skills.str.split('|')).explode('skill')
top = sk.skill.value_counts().head(30).index
sk[sk.skill.isin(top)].groupby('skill').salary_mid_usd.describe(percentiles=[.25, .5, .75])[['count', '25%', '50%', '75%']].sort_values('50%', ascending=False)

In [ ]:
df.pivot_table(index='role_family', columns='region_group', values='salary_mid_usd', aggfunc='median').round(-2)

## Get the full record from the Radar API

The sample has one row per posting with pay. The salary-benchmark endpoint returns count, median and range for any title, city, currency and skill set across the full, daily-updated postings. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['job_title'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/jobs/salary-benchmark', headers={'Authorization': f'Bearer {key}'},
                      json={'country': 'US', 'currency': 'USD', 'title': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp